## Setup

In [127]:
import numpy as np                      # imports the Numpy library for numerical tools
import pandas as pd                     # imports the Pandas library for data manipulation and analysis

# File management
from pathlib import Path                # imports the Path class from the pathlib library for handling file paths
import os                               # imports the OS library for interacting with the operating system  

def find_project_root(start=Path.cwd(), marker='data'): # defines a function to find the project root directory by looking for a specific marker (default is 'data')
    current = start
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()

os.chdir(PROJECT_ROOT) # changes the current working directory to the project root directory so that all file operations are relative to this directory

# tools for text processing
from tools import dataframe_tools as dftools


## Importing and formating dataframe

In [128]:
df_works = pd.read_json('data/bronze/dataframes/works.json')
df_works = df_works.drop(columns=['full_information'])
df_fiction = pd.read_csv('data/bronze/dataframes/lovecraft_fiction.csv')

cols = [col for col in df_works.columns]
df_works[cols] = df_works[cols].apply(lambda x: x.str.strip().str.lower() if x.dtype == "object" else x)

In [129]:
df_works

,title,magazine,year_published
0,the alchemist.,the united amateur,1916
1,at the mountains of madness.,astounding stories,1936
2,azathoth.,leaves,1938
3,the beast in the cave.,the vagrant,1918
4,beyond the wall of sleep.,pine cones,1919
...,...,...,...
94,"rimel, duane w. ""the tree on the hill.""",polaris,1940
95,"with sterling, kenneth. ""in the walls of eryx.""",weird tales,1939
96,"talman, wilfred blanch. ""two black bottles.""",weird tales,1927
97,"whitehead, henry s. ""bothon.""",amazing stories,1946


There are some tales that the name of the colaborators where included into the title during the conversion of the .json. We now remove those and create a separate column for colaborations.

In [130]:
collaborators = []
titles = []

for title in df_works['title']:
    if '"' in title:
        collaborator = title.split('"')[0].strip()
        collaborators.append(collaborator)
        titles.append(title.split('"')[1].strip())
    else:
        collaborators.append(None)
        titles.append(title)
        
df_works['collaboration'] = collaborators
df_works['title'] = titles 

We now normalize all the column contents

In [131]:
cols = ['title', 'collaboration', 'magazine']
df_works[cols] = df_works[cols].apply(lambda col: col.map(dftools.normalize_title))

In [132]:
df_works

,title,magazine,year_published,collaboration
0,the_alchemist,the_united_amateur,1916,None
1,at_the_mountains_of_madness,astounding_stories,1936,None
2,azathoth,leaves,1938,None
3,the_beast_in_the_cave,the_vagrant,1918,None
4,beyond_the_wall_of_sleep,pine_cones,1919,None
...,...,...,...,...
94,the_tree_on_the_hill,polaris,1940,rimel_duane_w
95,in_the_walls_of_eryx,weird_tales,1939,with_sterling_kenneth
96,two_black_bottles,weird_tales,1927,talman_wilfred_blanch
97,bothon,amazing_stories,1946,whitehead_henry_s


The file $\tt lovecraft\_fiction$ has information about the stories form of publication. We now merge this into the original dataframe. But first we format $\tt lovecraft\_fiction$ so it has the same form as $\tt df\_works$

In [133]:
df_fiction.rename(columns={'Title': 'title','Form':'form'}, inplace=True)
df_fiction['title']= df_fiction['title'].apply(dftools.normalize_title)
df_fiction = df_fiction.drop(columns={'id','Date written', 'Date published'})

Now we merge both to create a dataframe with the form of the stories with its form of publication. 

In [134]:
df = pd.merge(df_works,df_fiction, on = ['title'],how='left')
df['form'] = df['form'].apply(dftools.normalize_title)
df


,title,magazine,year_published,collaboration,form
0,the_alchemist,the_united_amateur,1916,None,short_story
1,at_the_mountains_of_madness,astounding_stories,1936,None,novella
2,azathoth,leaves,1938,None,fragment
3,the_beast_in_the_cave,the_vagrant,1918,None,None
4,beyond_the_wall_of_sleep,pine_cones,1919,None,short_story
...,...,...,...,...,...
94,the_tree_on_the_hill,polaris,1940,rimel_duane_w,None
95,in_the_walls_of_eryx,weird_tales,1939,with_sterling_kenneth,None
96,two_black_bottles,weird_tales,1927,talman_wilfred_blanch,None
97,bothon,amazing_stories,1946,whitehead_henry_s,None


The rows with $\tt form = None$ must be added manually. Providing these references to NotebookLM, we got the following table *(that we still need to check)*

In [135]:
df_collab_form = pd.read_json('data/bronze/dataframes/collaboration_story_form.json')
df_collab_form

,title,form
0,the_beast_in_the_cave,short_story
1,poetry_and_the_gods,short_story
2,the_crawling_chaos,short_story
3,herbert_westreanimator_i_from_the_dark,short_story
4,the_horror_at_martins_beach,short_story
5,ashes,short_story
6,under_the_pyramids,short_story
7,the_ghosteater,short_story
8,the_loved_dead,short_story
9,deaf_dumb_and_blind,short_story


Now we merge this with df

In [136]:
df = pd.merge(df,df_collab_form,on='title',how='left',suffixes=('','_y'))
df['form'] = df['form'].combine_first(df['form_y'])
df = df.drop(columns=['form_y'])


In [137]:
df = df.sort_values(by='year_published',ignore_index=True)
df.to_csv('data/bronze/dataframes/lovecraft_works.csv', index=False)

In [138]:
df

,title,magazine,year_published,collaboration,form
0,the_alchemist,the_united_amateur,1916,None,short_story
1,a_reminiscence_of_dr_samuel_johnson,the_united_amateur,1917,None,short_story
2,the_beast_in_the_cave,the_vagrant,1918,None,short_story
3,beyond_the_wall_of_sleep,pine_cones,1919,None,short_story
4,the_white_ship,the_united_amateur,1919,None,short_story
...,...,...,...,...,...
94,bothon,amazing_stories,1946,whitehead_henry_s,short_story
95,the_dreamquest_of_unknown_kadath,the_arkham_sampler,1948,None,novella
96,sweet_ermengarde_or_the_heart_of_a_country_girl,mirage,1965,None,short_story
97,ex_oblivione,magazine_of_horror,1968,None,short_story
